# Rwanda EO ML Lab 1

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

---

### Running this notebook

| Method | Steps |
|---|---|
| **Google Colab (browser)** | Upload to Google Drive → right-click → *Open with → Google Colaboratory* |
| **VS Code + Colab runtime** | 1. Open Colab in browser and start a session<br>2. In Colab: **Connect** dropdown → **Connect to a local runtime** (or copy the runtime URL from the address bar)<br>3. In VS Code kernel picker: **Select Kernel → Existing Jupyter Server** → paste the Colab runtime URL |

> **Note:** Google Earth Engine authentication (`ee.Authenticate()`) will open a browser window. In Colab this works automatically; in VS Code connected to a Colab runtime you may need to follow the printed link manually.

In [1]:
# Cell 1: Install required packages (auto-installs in Google Colab)
import sys

if 'google.colab' in sys.modules:
    import subprocess
    for pkg in ['geemap', 'earthengine-api']:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)
    print('Colab packages installed.')
else:
    print('Not in Colab — ensure geemap and earthengine-api are installed locally.')

import ee
import geemap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import seaborn as sns

print('All packages imported successfully')

All packages imported successfully


In [ ]:
# (Install logic has been merged into Cell 1 above)

Authenticate Authorization


In [2]:
#Cell 2: Authenticate GEE
ee.Authenticate()
ee.Initialize(project='gee-mymaap')
print('GEE initialised:', ee.String('GEE connection successful').getInfo())

GEE initialised: GEE connection successful


Define Your Study Area (Kigali ROI)


In [3]:
# Cell 3: Define study area covering Kigali and surroundings
# Coordinates: [west, south, east, north] bounding box around Kigali Province

roi = ee.Geometry.Rectangle([29.85, -2.10, 30.25, -1.80])

# Alternative: use FAO GAUL administrative boundaries
# rwanda = ee.FeatureCollection('FAO/GAUL/2015/level1')
# .filter(ee.Filter.eq('ADM0_NAME', 'Rwanda'))
# roi = rwanda.geometry()

print('ROI area (km2):', roi.area().divide(1e6).getInfo())

ROI area (km2): 1482.8658382861486


In [4]:
#Cell 4: Acquire Sentinel-2 SR imagery (dry season composite)
s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(roi)
    .filterDate('2022-06-01', '2022-09-30')
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10))
    .select(['B2', 'B3', 'B4', 'B5', 'B8', 'B8A', 'B11', 'B12'])
    .median() # cloud-free composite
    .clip(roi))

#Scale reflectance values from DN to [0, 1]
s2 = s2.divide(10000)

print('Available bands:', s2.bandNames().getInfo())
print('Image CRS:', s2.projection().crs().getInfo())

Available bands: ['B2', 'B3', 'B4', 'B5', 'B8', 'B8A', 'B11', 'B12']
Image CRS: EPSG:4326


Load and Filter Sentinel-2


In [5]:
#Cell 5: Interactive map with true-colour composite
Map = geemap.Map()
Map.centerObject(roi, zoom=11)

#True colour RGB (B4 Red, B3 Green, B2 Blue)
vis_rgb = {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 0.25, 'gamma': 1.4}

# False colour infrared (B8-NIR, B4-Red, B3-Green) vegetation bright red
vis_fci = {'bands': ['B8', 'B4', 'B3'], 'min': 0, 'max': 0.5, 'gamma': 1.4}

Map.addLayer(s2, vis_rgb, 'Kigali RGB 2022')
Map.addLayer(s2, vis_fci, 'False Colour Infrared', shown=False)
Map.addLayer(roi, {}, 'Study Area ROI')
Map

Map(center=[-1.9500074140417862, 30.05000000000008], controls=(WidgetControl(options=['position', 'transparent…

Compute spectral indices on the Sentinel-2 composite


In [6]:

# NDVI: Normalised Difference Vegetation Index
ndvi = s2.normalizedDifference(['B8', 'B4']).rename('NDVI')

# NDWI: Normalised Difference Water Index (Gao 1996)
ndwi = s2.normalizedDifference(['B3', 'B8']).rename('NDWI')

# NDBI: Normalised Difference Built-up Index
ndbi = s2.normalizedDifference(['B11', 'B8']).rename('NDBI')

# EVI: Enhanced Vegetation Index
evi = s2.expression(
    '2.5 * (NIR - Red) / (NIR + 6 * Red - 7.5 * Blue + 1)',
    {'NIR': s2.select('B8'), 'Red': s2.select('B4'), 'Blue': s2.select('B2')}
).rename('EVI')

# SAVI: Soil-Adjusted Vegetation Index (L=0.5)
savi = s2.expression(
    '1.5 * (NIR - Red) / (NIR + Red + 0.5)',
    {'NIR': s2.select('B8'), 'Red': s2.select('B4')}
).rename('SAVI')

# BSI: Bare Soil Index
bsi = s2.expression(
    '((SWIR + Red) - (NIR + Blue)) / ((SWIR + Red) + (NIR + Blue))',
    {'SWIR': s2.select('B11'), 'Red': s2.select('B4'),
     'NIR': s2.select('B8'), 'Blue': s2.select('B2')}
).rename('BSI')

# Stack all indices into a multi-band image
indices = ndvi.addBands([ndwi, ndbi, evi, savi, bsi])
print('Index bands:', indices.bandNames().getInfo())

Index bands: ['NDVI', 'NDWI', 'NDBI', 'EVI', 'SAVI', 'BSI']


Visualise each index on the interactive map

In [7]:
#Cell 7: Visualise each index on the interactive map
ndvi_vis = {'min': -0.2, 'max': 0.8, 'palette': ['#d73027','#ffffbf','#1a9850']}
ndwi_vis = {'min': -0.3, 'max': 0.5, 'palette': ['#a50026','#7f7f7f','#313695']}
ndbi_vis = {'min': -0.5, 'max': 0.5, 'palette': ['#1a9850','#ffffbf','#d73027']}

Map2 = geemap.Map()
Map2.centerObject(roi, zoom=11)

Map2.addLayer(ndvi, ndvi_vis, 'NDVI (green veg, red bare)')
Map2.addLayer(ndwi, ndwi_vis, 'NDWI (blue water)', shown=False)
Map2.addLayer(ndbi, ndbi_vis, 'NDBI (red urban)', shown=False)

Map2

Map(center=[-1.9500074140417862, 30.05000000000008], controls=(WidgetControl(options=['position', 'transparent…

Export Sentinel-2 composite to Google Drive (LO 31)

In [8]:
#Cell 8: Export Sentinel-2 composite to Google Drive (LO 31)
# This creates a GeoTIFF in your Google Drive > Rwanda_EO_Lab folder
export_s2 = ee.batch.Export.image.toDrive(
    image = s2.select(['B2', 'B3', 'B4', 'B8', 'B11']),
    description = 'Sentinel2_Kigali_2022_DrySeasonComposite',
    folder = 'Rwanda_EO_Lab',
    fileNamePrefix = 'S2_Kigali_2022',
    region = roi,
    scale = 10, # 10m resolution
    crs = 'EPSG:32735', # WGS 84 / UTM zone 35S
    maxPixels = 1e9
)
export_s2.start()

# Export NDVI separately
export_ndvi = ee.batch.Export.image.toDrive(
    image = ndvi,
    description = 'NDVI_Kigali_2022',
    folder = 'Rwanda_EO_Lab',
    fileNamePrefix = 'NDVI_Kigali_2022',
    region = roi,
    scale = 10,
    crs = 'EPSG:32735',
    maxPixels = 1e9
)
export_ndvi.start()

print('Export tasks started. Check status at: https://code.earthengine.google.com/tasks')

Export tasks started. Check status at: https://code.earthengine.google.com/tasks


#Lab Session 2: ML Classification!

Build the feature stack (8 S2 bands + 6 spectral indices = 14 features)

In [9]:
#Cell 9: Build the feature stack (8 S2 bands + 6 spectral indices = 14 features)
# This is the input to all ML classifiers
feature_stack = s2.addBands(indices) # 14-band image

# Define Rwanda land cover class names and integer labels
class_names = {
    1: 'Dense Forest',     # Nyungwe, Gishwati-Mukura
    2: 'Cropland',         # Terraced agriculture
    3: 'Urban',            # Kigali built-up
    4: 'Open Water',       # Lakes, rivers
    5: 'Grassland/Shrub',  # Degraded land, hillside shrub
    6: 'Wetland',          # Papyrus swamps
    7: 'Bare Soil'         # Laterite, erosion scars
}

print('Feature stack bands:', feature_stack.bandNames().getInfo())
print('Land cover classes:', class_names)

Feature stack bands: ['B2', 'B3', 'B4', 'B5', 'B8', 'B8A', 'B11', 'B12', 'NDVI', 'NDWI', 'NDBI', 'EVI', 'SAVI', 'BSI']
Land cover classes: {1: 'Dense Forest', 2: 'Cropland', 3: 'Urban', 4: 'Open Water', 5: 'Grassland/Shrub', 6: 'Wetland', 7: 'Bare Soil'}


Training sample points for 7 Rwanda land cover classes


In [10]:
#Cell 10: Training sample points for 7 Rwanda land cover classes
#Each FeatureCollection contains manually verified reference points
#Coordinates based on known land cover locations in Rwanda

#Dense Forest samples (Nyungwe Forest National Park area)
forest_pts = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Point([29.28, -2.49]), {'class': 1}),
    ee.Feature(ee.Geometry.Point([29.30, -2.55]), {'class': 1}),
    ee.Feature(ee.Geometry.Point([29.25, -2.52]), {'class': 1}),
    ee.Feature(ee.Geometry.Point([29.35, -2.48]), {'class': 1}),
    ee.Feature(ee.Geometry.Point([29.22, -2.45]), {'class': 1})
])

# Cropland samples (terraced hillsides across Rwanda)
crop_pts = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Point([29.95, -1.95]), {'class': 2}),
    ee.Feature(ee.Geometry.Point([30.10, -2.00]), {'class': 2}),
    ee.Feature(ee.Geometry.Point([30.00, -2.10]), {'class': 2}),
    ee.Feature(ee.Geometry.Point([29.90, -2.05]), {'class': 2}),
    ee.Feature(ee.Geometry.Point([30.15, -2.15]), {'class': 2})
])

#Urban samples (Kigali city centre)
urban_pts = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Point([30.062, -1.944]), {'class': 3}),
    ee.Feature(ee.Geometry.Point([30.070, -1.960]), {'class': 3}),
    ee.Feature(ee.Geometry.Point([30.055, -1.955]), {'class': 3}),
    ee.Feature(ee.Geometry.Point([30.075, -1.948]), {'class': 3}),
    ee.Feature(ee.Geometry.Point([30.080, -1.935]), {'class': 3})
])

# Water samples (Lake Muhazi, Lake Kivu)
water_pts = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Point([30.22, -1.92]), {'class': 4}),
    ee.Feature(ee.Geometry.Point([30.24, -1.90]), {'class': 4}),
    ee.Feature(ee.Geometry.Point([29.30, -2.15]), {'class': 4}),
    ee.Feature(ee.Geometry.Point([29.25, -2.20]), {'class': 4})
])

# Grassland / Shrub samples
grass_pts = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Point([30.18, -1.85]), {'class': 5}),
    ee.Feature(ee.Geometry.Point([30.12, -1.80]), {'class': 5}),
    ee.Feature(ee.Geometry.Point([30.20, -1.88]), {'class': 5})
])

# Wetland samples
wetland_pts = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Point([30.15, -2.05]), {'class': 6}),
    ee.Feature(ee.Geometry.Point([30.13, -2.08]), {'class': 6})
])

#Bare soil samples (laterite, erosion scars)
bare_pts = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Point([30.05, -2.02]), {'class': 7}),
    ee.Feature(ee.Geometry.Point([30.08, -2.00]), {'class': 7}),
    ee.Feature(ee.Geometry.Point([30.10, -1.98]), {'class': 7})
])

#Merge all classes into one FeatureCollection
training_pts = (forest_pts.merge(crop_pts).merge(urban_pts)
                .merge(water_pts).merge(grass_pts)
                .merge(wetland_pts).merge(bare_pts))

print('Total training points:', training_pts.size().getInfo())

Total training points: 27


Sample feature values at all training points

In [11]:
#Cell 11: Sample feature values at all training points
feature_names = ['B2', 'B3', 'B4', 'B5', 'B8', 'B8A', 'B11', 'B12',
                 'NDVI', 'NDWI', 'NDBI', 'EVI', 'SAVI', 'BSI']

sampled = feature_stack.sampleRegions(
    collection = training_pts,
    properties = ['class'],
    scale = 10,
    geometries = False
)

# Convert to pandas DataFrame for scikit-learn
sample_list = sampled.getInfo()['features']
records = [f['properties'] for f in sample_list]
df = pd.DataFrame(records)

print('Training dataframe shape:', df.shape)
print(df.head())
print('Class distribution:')
print(df['class'].value_counts().sort_index())

Training dataframe shape: (19, 15)
      B11     B12      B2      B3      B4      B5      B8     B8A       BSI  \
0  0.2438  0.1740  0.0484  0.0812  0.1220  0.1562  0.2254  0.2291  0.143840   
1  0.2069  0.1215  0.0331  0.0541  0.0564  0.0930  0.2196  0.2380  0.020543   
2  0.2492  0.1821  0.0433  0.0666  0.0723  0.1242  0.2440  0.2722  0.056176   
3  0.2707  0.1813  0.0372  0.0636  0.0656  0.1288  0.2685  0.2993  0.047664   
4  0.2141  0.1576  0.0720  0.0987  0.1046  0.1394  0.2227  0.2272  0.039126   

        EVI      NDBI      NDVI      NDWI      SAVI  class  
0  0.162130  0.039216  0.297640 -0.470320  0.183030      2  
1  0.311510 -0.029777  0.591304 -0.604677  0.315464      2  
2  0.317246  0.010543  0.542839 -0.571153  0.315509      2  
3  0.366749  0.004080  0.607303 -0.616983  0.364884      2  
4  0.225330 -0.019689  0.360831 -0.385812  0.214130      3  
Class distribution:
class
2    4
3    5
4    2
5    3
6    2
7    3
Name: count, dtype: int64


Prepare X (features) and y (labels) arrays

In [12]:
#Cell 12: Prepare X (features) and y (labels) arrays
X = df[feature_names].values.astype(float)
y = df['class'].values.astype(int)

# Handle any NaN values (can occur at cloud-masked edges)
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='median')
X = imputer.fit_transform(X)

# Feature scaling: REQUIRED for SVM and KNN; optional but harmless for RF
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f'X shape: {X.shape}, y shape: {y.shape}')
print(f'Classes: {np.unique(y)}')

X shape: (19, 14), y shape: (19,)
Classes: [2 3 4 5 6 7]


5-fold stratified cross-validation for all three classifiers

In [ ]:
#Cell 13: 5-fold stratified cross-validation for all three classifiers
# StratifiedKFold preserves class proportions in each fold
# MENTOR FIX: Changed n_splits=5 to n_splits=2 so it doesn't crash on our 19 points!
cv = StratifiedKFold(n_splits=2, shuffle=True, random_state=42)

classifiers = {
    'Random Forest': RandomForestClassifier(n_estimators=200, max_features='sqrt',
                                            class_weight='balanced', random_state=42, n_jobs=-1),
    'SVM (RBF)': SVC(kernel='rbf', C=10, gamma='scale',
                     class_weight='balanced', random_state=42),
    'KNN (k=5)': KNeighborsClassifier(n_neighbors=5, metric='euclidean', n_jobs=-1)
}

print('Cross-validation results (Overall Accuracy):')
cv_results = {}

for name, clf in classifiers.items():
    # Use scaled features for SVM and KNN, unscaled for RF
    X_in = X_scaled if name != 'Random Forest' else X
    scores = cross_val_score(clf, X_in, y, cv=cv, scoring='accuracy', n_jobs=-1)
    cv_results[name] = scores
    print(f'  {name:<20}: {scores.mean():.3f} +/- {scores.std():.3f}')

Cross-validation results (Overall Accuracy):


In [ ]:
#Cell 14: Final training + evaluation using 80/20 split
from sklearn.model_selection import train_test_split

# MENTOR FIX: Removed stratify=y so it doesn't crash on our 4-point test set!
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

#Train Random Forest (best performer in most Rwanda EO studies)
rf = RandomForestClassifier(n_estimators=300, max_features='sqrt',
                            class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

# Print full classification report
# MENTOR FIX: Dynamically fetch names only for the classes that actually made it into the test set
import numpy as np
labels_in_test = np.unique(np.concatenate((y_test, y_pred)))
target_names = [class_names[i] for i in labels_in_test]
print(classification_report(y_test, y_pred, labels=labels_in_test, target_names=target_names))

#Cohen's Kappa
from sklearn.metrics import cohen_kappa_score
kappa = cohen_kappa_score(y_test, y_pred)
print(f'Cohen Kappa: {kappa:.3f}')

Confusion matrix visualisation

In [ ]:
#Cell 15: Confusion matrix visualisation
fig, ax = plt.subplots(figsize=(9,7))

# MENTOR FIX: Force the matrix to use the exact labels that appeared in our small test set
cm = confusion_matrix(y_test, y_pred, labels=labels_in_test)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_names)

disp.plot(ax=ax, cmap='Blues', colorbar=True)
ax.set_title('Random Forest Confusion Matrix\nKigali Rwanda Land Cover 2022', fontsize=13, fontweight='bold')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.savefig('RF_confusion_matrix_Rwanda.png', dpi=150, bbox_inches='tight')
plt.show()

print('Matrix saved to RF_confusion_matrix_Rwanda.png')

Feature importance ranking (LO 24 assessing algorithm behaviour)

In [ ]:
#Cell 16: Feature importance ranking (LO 24 assessing algorithm behaviour)
importances = pd.Series(rf.feature_importances_, index=feature_names)
importances_sorted = importances.sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10,5))
importances_sorted.plot.bar(ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Random Forest Feature Importance\nKigali Rwanda Land Cover Classification',
             fontsize=12, fontweight='bold')
ax.set_ylabel('Importance Score (mean decrease in impurity)')
ax.set_xlabel('Feature')
plt.xticks(rotation=40, ha='right')
plt.tight_layout()
plt.savefig('RF_feature_importance_Rwanda.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 3 features:', importances_sorted.head(3).index.tolist())